In [1]:
import warnings
warnings.filterwarnings('ignore')

(Tutorial_Group_Name_Code)=
# Getting a one-letter group code

*Projecting a group name under its declared source convention.*

A label can mean different compounds in a sequence alphabet, a force-field vocabulary and a chemical dictionary. Choose the source convention before projecting an amino-acid parent; the operation preserves the original group and its chemistry.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

{func}`molsysmt.element.group.get_1_letter_code_from_name` documents the scalar arguments and refusal cases.
:::

## Choosing a convention

Let's show how this function works with reviewed group labels. The scalar helper receives a name, not a molecular-system form. Use `sequence` for the shared amino-acid alphabet, `amber` for the supported peptide-state labels, `ccd` for the bounded component snapshot, and `legacy` only for the historical parent table. The returned letter is a string, not a physical quantity.

In [2]:
import molsysmt as msm

In [3]:
for name, scheme in [('ALA', None), ('GLH', 'amber'), ('GLH', 'legacy'), ('XLE', 'sequence'), ('MSE', 'ccd')]:
    letter = msm.element.group.get_1_letter_code_from_name(name, naming_scheme=scheme)
    print(name, scheme, letter)

ALA None A
GLH amber E
GLH legacy Q
XLE sequence J
MSE ccd M


## Refusing ambiguous meanings

With `naming_scheme=None`, a context-dependent name requires an explicit choice. Under CCD, XLE is a non-peptide compound and has no amino-acid letter. It must not be translated into the sequence ambiguity symbol J. Unsupported names never fall back to another declared convention.

In [4]:
for scheme in [None, 'ccd']:
    try:
        msm.element.group.get_1_letter_code_from_name('XLE', naming_scheme=scheme)
    except msm.ArgumentError as error:
        print(scheme, 'refused at', error.extra['argument'])

None refused at naming_scheme
ccd refused at name


## Reading stored declarations

For a molecular system, declare the convention per group and use the native sequence conversion. Canonical sequence construction knows its alphabet; noncanonical labels remain unknown until explicitly assigned. No atoms, coordinates or chemical state are created by the following sequence-only conversion.

In [5]:
molsys = msm.convert('amino_acids_3:AlaGlhHid', to_form='molsysmt.MolSys')
print(molsys.topology.group_naming_schemes)
molsys.topology.set_group_naming_schemes('amber', group_indices=[1, 2])
print(msm.convert(molsys, to_form='string:amino_acids_1'))
print(molsys.topology.groups.group_name.tolist())

('sequence', None, None)


AEH
['ALA', 'GLH', 'HID']


## Interpreting a source during conversion

Use `from_naming_scheme` when the input vocabulary is known but not stored. A one-letter query reads selected group positions in the requested order without changing the input. A native molecular output retains the declaration. A contradiction with stored context raises an error instead of replacing it. See {ref}`Tutorial_Convert`.

In [6]:
molsys_A = msm.convert('amino_acids_3:GlhHid', to_form='molsysmt.Topology')
print(msm.convert(molsys_A, to_form='string:amino_acids_1',
                  group_indices=[1, 0, 1], from_naming_scheme='amber'))
print(molsys_A.group_naming_schemes)
molsys_B = msm.convert(molsys_A, to_form='molsysmt.MolSys',
                       from_naming_scheme='amber')
print(molsys_B.topology.group_naming_schemes)
try:
    msm.convert(molsys_B, to_form='string:amino_acids_1',
                from_naming_scheme='legacy')
except Exception as error:
    print(error.extra['argument'])

HEH
(None, None)
('amber', 'amber')
from_naming_scheme


## Comparing a declared sequence

Sequence identity reads the stored convention and returns original source group indices. The projected letters describe a comparison; they do not replace the chemical groups. Biopython is optional and checked for its selected engine. See {ref}`Tutorial_Get_sequence_identity`.

In [7]:
identity, query_groups, reference_groups = msm.topology.get_sequence_identity(
    molsys_B, reference_molecular_system='amino_acids_1:EH')
print(identity, query_groups, reference_groups)

100.0 [0, 1] [0, 1]


## Retaining conventions on disk

H5MSM 0.5 preserves the group-aligned declaration registry and names. Loading a file does not infer a convention for missing metadata. A changed declaration invalidates attached interaction analyses; old analysis objects remain snapshots and coordinates are not copied.

In [8]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory(prefix='molsysmt-naming-example-') as task_directory:
    filename = str(Path(task_directory) / 'sequence.h5msm')
    msm.convert(molsys, to_form=filename)
    restored = msm.convert(filename, to_form='molsysmt.MolSys')
    print(restored.topology.group_naming_schemes)
    print(msm.convert(restored, to_form='string:amino_acids_1'))

('sequence', 'amber', 'amber')
AEH


## Understanding coverage

The sequence alphabet comes from DDBJ. The bounded Amber and CCD definitions retain their reviewed references; legacy selects the existing historical MDTraj-derived table. Projecting MSE to M does not replace selenium, add hydrogens or parameterize a molecule. Optional Ackredit records the data references actually used; it does not claim that an Amber or MDTraj engine ran.

:::{warning}

This is the scheme-aware helper at `msm.element.group`. The older helpers under `amino_acid` and other recognition/preparation consumers still await the coordinated cutover under [#549](https://github.com/uibcdf/molsysmt/issues/549).
:::

:::{seealso}
:class: dropdown

- {ref}`user-foundations-group-naming-schemes` compares definitions, sources and defaults.
- {ref}`user-foundations-native-world-classes-molsysmt-topology` explains per-group declarations.
- {ref}`user-tools-form-h5msm-05` describes versioned persistence.
:::